# Analytics and visualization

Compute return/risk, trade and regime diagnostics, then inspect equity curves and individual trade paths.

**Part 3/4 of a shared-state notebook series.** This part uses variables/functions from earlier parts. Opening another notebook normally creates a new kernel. Use `python scripts/run_series.py all_for_one --export outputs/all_for_one_combined.ipynb` from the repository root, then open the exported notebook for interactive execution. Exporting does not execute the research or restore missing data.

**Status:** curated research source; original saved outputs preserved. Original notes about results are historical observations, not revalidated conclusions. Read `docs/DATA_AND_RUNNING.md` before execution.

Source: `ALL_FOR_ONE.ipynb`, original cells 132–198 (zero-based).


# Performance Analytics Engine

In [112]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd

In [113]:
@dataclass(frozen=True)
class PerformanceConfig:
    # Crypto เทรดตลอดทั้งปี
    annualization_days: float = 365.25

    risk_free_rate_annual: float = 0.0

    min_return_observations: int = 30

    # หากต้องการ override จำนวน bars ต่อปี
    periods_per_year: Optional[float] = None

In [114]:
@dataclass
class PerformanceReport:
    strategy_id: str
    execution_timeframe: str

    summary: pd.Series
    trade_summary: pd.Series

    side_summary: pd.DataFrame
    yearly_summary: pd.DataFrame
    exit_reason_summary: pd.DataFrame

    regime_attribution: dict[str, pd.DataFrame]

    drawdown: pd.DataFrame

In [115]:
def infer_periods_per_year(
    execution_timeframe: str,
    annualization_days: float = 365.25,
) -> float:
    bar_duration = pd.Timedelta(
        execution_timeframe
    )

    if bar_duration <= pd.Timedelta(0):
        raise ValueError(
            "execution_timeframe must be positive."
        )

    year_duration = pd.Timedelta(
        days=annualization_days
    )

    return float(
        year_duration / bar_duration
    )

In [116]:
def safe_ratio(
    numerator: float,
    denominator: float,
) -> float:
    if (
        pd.isna(numerator)
        or pd.isna(denominator)
        or denominator == 0
    ):
        return np.nan

    return float(numerator / denominator)

In [117]:
def calculate_profit_factor(
    pnl: pd.Series,
) -> float:
    pnl = pd.to_numeric(
        pnl,
        errors="coerce",
    ).dropna()

    if pnl.empty:
        return np.nan

    gross_profit = float(
        pnl.loc[pnl > 0].sum()
    )

    gross_loss = float(
        -pnl.loc[pnl < 0].sum()
    )

    if gross_loss == 0:
        if gross_profit > 0:
            return np.inf

        return np.nan

    return gross_profit / gross_loss

In [118]:
def calculate_payoff_ratio(
    pnl: pd.Series,
) -> float:
    winners = pnl.loc[pnl > 0]
    losers = pnl.loc[pnl < 0]

    if winners.empty or losers.empty:
        return np.nan

    return float(
        winners.mean()
        / abs(losers.mean())
    )

In [119]:
def calculate_max_consecutive_runs(
    pnl: pd.Series,
) -> tuple[int, int]:
    max_wins = 0
    max_losses = 0

    current_wins = 0
    current_losses = 0

    for value in pnl.dropna():
        if value > 0:
            current_wins += 1
            current_losses = 0

            max_wins = max(
                max_wins,
                current_wins,
            )

        elif value < 0:
            current_losses += 1
            current_wins = 0

            max_losses = max(
                max_losses,
                current_losses,
            )

        else:
            current_wins = 0
            current_losses = 0

    return max_wins, max_losses

In [120]:
def prepare_performance_bars(
    result: ExecutionResult,
    cfg: PerformanceConfig,
) -> tuple[pd.DataFrame, float]:
    d = result.bars.copy()

    required = {
        "mtm_equity",
        "realized_equity",
        "unrealized_pnl",
        "position_sign",
        "position_notional",
        "gross_exposure",
    }

    missing = required.difference(d.columns)

    if missing:
        raise ValueError(
            f"Missing performance columns: "
            f"{sorted(missing)}"
        )

    if d.empty:
        raise ValueError(
            "Execution result contains no bars."
        )

    if cfg.periods_per_year is None:
        periods_per_year = (
            infer_periods_per_year(
                result.execution_timeframe,
                cfg.annualization_days,
            )
        )
    else:
        periods_per_year = float(
            cfg.periods_per_year
        )

    equity = pd.to_numeric(
        d["mtm_equity"],
        errors="coerce",
    )

    previous_equity = equity.shift(1)

    strategy_return = (
        equity / previous_equity - 1.0
    )

    # รวมผลต่างจาก initial capital ถึง bar แรก
    strategy_return.iloc[0] = (
        equity.iloc[0]
        / result.config.initial_capital
        - 1.0
    )

    # หาก equity ก่อนหน้าเป็นศูนย์หรือติดลบ
    # percentage return จะไม่มีความหมาย
    strategy_return = strategy_return.where(
        previous_equity.gt(0)
        | previous_equity.isna()
    )

    risk_free_per_bar = (
        (1.0 + cfg.risk_free_rate_annual)
        ** (1.0 / periods_per_year)
        - 1.0
    )

    d["strategy_return"] = strategy_return
    d["excess_return"] = (
        d["strategy_return"]
        - risk_free_per_bar
    )

    d["equity_peak"] = (
        d["mtm_equity"].cummax()
    )

    d["drawdown"] = (
        d["mtm_equity"]
        / d["equity_peak"]
        - 1.0
    )

    d["underwater"] = (
        d["drawdown"] < 0
    )

    d["long_exposure_flag"] = (
        d["position_sign"] > 0
    )

    d["short_exposure_flag"] = (
        d["position_sign"] < 0
    )

    d["exposure_flag"] = (
        d["position_sign"] != 0
    )

    return d, periods_per_year

In [121]:
def calculate_drawdown_duration_series(
    underwater: pd.Series,
) -> pd.Series:
    durations = np.zeros(
        len(underwater),
        dtype=np.int64,
    )

    current_duration = 0

    for i, is_underwater in enumerate(
        underwater.fillna(False).to_numpy()
    ):
        if is_underwater:
            current_duration += 1
        else:
            current_duration = 0

        durations[i] = current_duration

    return pd.Series(
        durations,
        index=underwater.index,
        dtype="Int64",
    )

In [122]:
def calculate_drawdown_diagnostics(
    performance_bars: pd.DataFrame,
    execution_timeframe: str,
) -> tuple[pd.DataFrame, dict[str, object]]:
    d = performance_bars[
        [
            "mtm_equity",
            "equity_peak",
            "drawdown",
            "underwater",
        ]
    ].copy()

    d["drawdown_duration_bars"] = (
        calculate_drawdown_duration_series(
            d["underwater"]
        )
    )

    bar_duration = pd.Timedelta(
        execution_timeframe
    )

    d["drawdown_duration"] = (
        d["drawdown_duration_bars"]
        * bar_duration
    )

    if d["drawdown"].dropna().empty:
        return d, {
            "max_drawdown": np.nan,
            "max_drawdown_peak_time": pd.NaT,
            "max_drawdown_trough_time": pd.NaT,
            "max_drawdown_recovery_time": pd.NaT,
            "max_drawdown_duration_bars": 0,
            "max_drawdown_duration": pd.Timedelta(0),
        }

    trough_time = d["drawdown"].idxmin()
    max_drawdown = float(
        d.loc[trough_time, "drawdown"]
    )

    equity_until_trough = d.loc[
        :trough_time,
        "mtm_equity",
    ]

    peak_time = equity_until_trough.idxmax()
    peak_equity = float(
        d.loc[peak_time, "mtm_equity"]
    )

    recovery_candidates = d.loc[
        trough_time:,
        "mtm_equity",
    ]

    recovery_candidates = (
        recovery_candidates.loc[
            recovery_candidates >= peak_equity
        ]
    )

    if recovery_candidates.empty:
        recovery_time = pd.NaT
    else:
        recovery_time = (
            recovery_candidates.index[0]
        )

    max_duration_bars = int(
        d["drawdown_duration_bars"].max()
    )

    diagnostics = {
        "max_drawdown": max_drawdown,
        "max_drawdown_peak_time": peak_time,
        "max_drawdown_trough_time": (
            trough_time
        ),
        "max_drawdown_recovery_time": (
            recovery_time
        ),
        "max_drawdown_duration_bars": (
            max_duration_bars
        ),
        "max_drawdown_duration": (
            max_duration_bars
            * bar_duration
        ),
    }

    return d, diagnostics

In [123]:
def calculate_return_risk_metrics(
    performance_bars: pd.DataFrame,
    *,
    initial_capital: float,
    periods_per_year: float,
    annualization_days: float,
    min_return_observations: int,
) -> dict[str, float]:
    equity = (
        performance_bars["mtm_equity"]
        .dropna()
    )

    returns = (
        performance_bars["strategy_return"]
        .dropna()
    )

    excess_returns = (
        performance_bars["excess_return"]
        .dropna()
    )

    if equity.empty:
        raise ValueError(
            "No valid equity observations."
        )

    final_equity = float(
        equity.iloc[-1]
    )

    total_return = (
        final_equity
        / initial_capital
        - 1.0
    )

    start_time = performance_bars.index[0]
    end_time = performance_bars.index[-1]

    duration_days = (
        end_time - start_time
    ) / pd.Timedelta(days=1)

    duration_years = (
        duration_days / annualization_days
    )

    if (
        duration_years > 0
        and initial_capital > 0
        and final_equity > 0
    ):
        cagr = (
            final_equity / initial_capital
        ) ** (1.0 / duration_years) - 1.0
    else:
        cagr = np.nan

    enough_returns = (
        len(excess_returns)
        >= min_return_observations
    )

    if enough_returns:
        return_mean = float(
            returns.mean()
        )

        return_std = float(
            returns.std(ddof=1)
        )

        excess_mean = float(
            excess_returns.mean()
        )

        excess_std = float(
            excess_returns.std(ddof=1)
        )

        annualized_mean_return = (
            return_mean * periods_per_year
        )

        annualized_volatility = (
            return_std
            * np.sqrt(periods_per_year)
        )

        sharpe = (
            excess_mean
            / excess_std
            * np.sqrt(periods_per_year)
            if excess_std > 0
            else np.nan
        )

        downside_returns = np.minimum(
            excess_returns.to_numpy(
                dtype=float
            ),
            0.0,
        )

        downside_deviation_per_bar = float(
            np.sqrt(
                np.mean(
                    downside_returns ** 2
                )
            )
        )

        annualized_downside_deviation = (
            downside_deviation_per_bar
            * np.sqrt(periods_per_year)
        )

        sortino = (
            excess_mean
            * periods_per_year
            / annualized_downside_deviation
            if annualized_downside_deviation > 0
            else np.nan
        )

    else:
        annualized_mean_return = np.nan
        annualized_volatility = np.nan
        sharpe = np.nan
        sortino = np.nan

    max_drawdown = float(
        performance_bars["drawdown"].min()
    )

    calmar = (
        cagr / abs(max_drawdown)
        if (
            pd.notna(cagr)
            and max_drawdown < 0
        )
        else np.nan
    )

    positive_bar_ratio = (
        float((returns > 0).mean())
        if len(returns) > 0
        else np.nan
    )

    negative_bar_ratio = (
        float((returns < 0).mean())
        if len(returns) > 0
        else np.nan
    )

    return {
        "initial_capital": (
            float(initial_capital)
        ),
        "final_mtm_equity": final_equity,
        "total_return": total_return,

        "duration_days": float(
            duration_days
        ),
        "duration_years": float(
            duration_years
        ),

        "cagr": cagr,
        "annualized_mean_return": (
            annualized_mean_return
        ),
        "annualized_volatility": (
            annualized_volatility
        ),
        "annualized_sharpe": sharpe,
        "annualized_sortino": sortino,
        "calmar_ratio": calmar,

        "max_drawdown": max_drawdown,

        "positive_bar_ratio": (
            positive_bar_ratio
        ),
        "negative_bar_ratio": (
            negative_bar_ratio
        ),
    }

In [124]:
def summarize_trade_dataframe(
    trades: pd.DataFrame,
) -> pd.Series:
    if trades.empty:
        return pd.Series(
            {
                "n_trades": 0,

                "n_winners": 0,
                "n_losers": 0,
                "n_flat": 0,

                "win_rate": np.nan,
                "loss_rate": np.nan,

                "total_net_pnl": 0.0,
                "mean_net_pnl": np.nan,
                "median_net_pnl": np.nan,

                "mean_trade_return": np.nan,
                "median_trade_return": np.nan,

                "profit_factor": np.nan,
                "payoff_ratio": np.nan,

                "best_trade_pnl": np.nan,
                "worst_trade_pnl": np.nan,

                "average_holding_bars": np.nan,
                "median_holding_bars": np.nan,

                "average_holding_duration": pd.NaT,
                "median_holding_duration": pd.NaT,

                "average_mfe_return": np.nan,
                "average_mae_return": np.nan,

                "max_consecutive_wins": 0,
                "max_consecutive_losses": 0,
            }
        )

    pnl = pd.to_numeric(
        trades["net_pnl"],
        errors="coerce",
    )

    trade_return = pd.to_numeric(
        trades["net_return_on_entry"],
        errors="coerce",
    )

    winners = pnl > 0
    losers = pnl < 0
    flat = pnl == 0

    max_wins, max_losses = (
        calculate_max_consecutive_runs(pnl)
    )

    return pd.Series(
        {
            "n_trades": int(len(trades)),

            "n_winners": int(winners.sum()),
            "n_losers": int(losers.sum()),
            "n_flat": int(flat.sum()),

            "win_rate": float(
                winners.mean()
            ),

            "loss_rate": float(
                losers.mean()
            ),

            "total_net_pnl": float(
                pnl.sum()
            ),

            "mean_net_pnl": float(
                pnl.mean()
            ),

            "median_net_pnl": float(
                pnl.median()
            ),

            "mean_trade_return": float(
                trade_return.mean()
            ),

            "median_trade_return": float(
                trade_return.median()
            ),

            "profit_factor": (
                calculate_profit_factor(pnl)
            ),

            "payoff_ratio": (
                calculate_payoff_ratio(pnl)
            ),

            "best_trade_pnl": float(
                pnl.max()
            ),

            "worst_trade_pnl": float(
                pnl.min()
            ),

            "average_holding_bars": float(
                trades["holding_bars"].mean()
            ),

            "median_holding_bars": float(
                trades["holding_bars"].median()
            ),

            "average_holding_duration": (
                trades["holding_duration"].mean()
            ),

            "median_holding_duration": (
                trades["holding_duration"].median()
            ),

            "average_mfe_return": float(
                trades[
                    "mfe_return_on_entry"
                ].mean()
            ),

            "median_mfe_return": float(
                trades[
                    "mfe_return_on_entry"
                ].median()
            ),

            "average_mae_return": float(
                trades[
                    "mae_return_on_entry"
                ].mean()
            ),

            "median_mae_return": float(
                trades[
                    "mae_return_on_entry"
                ].median()
            ),

            "max_consecutive_wins": (
                max_wins
            ),

            "max_consecutive_losses": (
                max_losses
            ),
        }
    )

In [125]:
def build_side_summary(
    trades: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for side in ["long", "short", "all"]:
        if side == "all":
            subset = trades
        else:
            subset = trades.loc[
                trades["side"] == side
            ]

        summary = (
            summarize_trade_dataframe(
                subset
            )
        )

        summary["side"] = side
        rows.append(summary)

    result = pd.DataFrame(rows)

    return result.set_index("side")

In [126]:
def calculate_execution_metrics(
    result: ExecutionResult,
    performance_bars: pd.DataFrame,
    duration_years: float,
) -> dict[str, float]:
    fills = result.fills

    if fills.empty:
        total_fees = 0.0
        total_slippage = 0.0
        traded_notional = 0.0
    else:
        total_fees = float(
            fills["fee"].sum()
        )

        total_slippage = float(
            fills["slippage_cost"].sum()
        )

        traded_notional = float(
            fills["notional"].abs().sum()
        )

    average_equity = float(
        performance_bars[
            "mtm_equity"
        ].mean()
    )

    turnover = safe_ratio(
        traded_notional,
        average_equity,
    )

    annualized_turnover = (
        turnover / duration_years
        if (
            pd.notna(turnover)
            and duration_years > 0
        )
        else np.nan
    )

    return {
        "n_decisions": int(
            len(result.decisions)
        ),

        "n_fills": int(
            len(result.fills)
        ),

        "n_closed_trades": int(
            len(result.trades)
        ),

        "has_open_position": (
            result.open_position is not None
        ),

        "total_fees": total_fees,
        "total_slippage_cost": (
            total_slippage
        ),

        "fee_drag_vs_initial": (
            total_fees
            / result.config.initial_capital
        ),

        "slippage_drag_vs_initial": (
            total_slippage
            / result.config.initial_capital
        ),

        "traded_notional": (
            traded_notional
        ),

        "turnover": turnover,
        "annualized_turnover": (
            annualized_turnover
        ),

        "exposure": float(
            performance_bars[
                "exposure_flag"
            ].mean()
        ),

        "long_exposure": float(
            performance_bars[
                "long_exposure_flag"
            ].mean()
        ),

        "short_exposure": float(
            performance_bars[
                "short_exposure_flag"
            ].mean()
        ),

        "average_gross_exposure": float(
            performance_bars[
                "gross_exposure"
            ].fillna(0.0).mean()
        ),

        "max_gross_exposure": float(
            performance_bars[
                "gross_exposure"
            ].fillna(0.0).max()
        ),

        "n_executed_entries": int(
            performance_bars[
                "executed_entry"
            ].sum()
        ),

        "n_executed_exits": int(
            performance_bars[
                "executed_exit"
            ].sum()
        ),

        "n_executed_reversals": int(
            performance_bars[
                "executed_reversal"
            ].sum()
        ),
    }

In [127]:
def calculate_period_sharpe(
    returns: pd.Series,
    periods_per_year: float,
    min_observations: int,
) -> float:
    returns = returns.dropna()

    if len(returns) < min_observations:
        return np.nan

    std = float(
        returns.std(ddof=1)
    )

    if std <= 0:
        return np.nan

    return float(
        returns.mean()
        / std
        * np.sqrt(periods_per_year)
    )

In [128]:
def calculate_period_drawdown(
    returns: pd.Series,
) -> float:
    returns = returns.fillna(0.0)

    cumulative = (
        1.0 + returns
    ).cumprod()

    peak = cumulative.cummax()

    drawdown = (
        cumulative / peak - 1.0
    )

    return float(
        drawdown.min()
    )

In [129]:
def build_yearly_summary(
    result: ExecutionResult,
    performance_bars: pd.DataFrame,
    periods_per_year: float,
    min_return_observations: int,
) -> pd.DataFrame:
    rows = []

    trades = result.trades.copy()

    if not trades.empty:
        trades["entry_year"] = (
            pd.to_datetime(
                trades["entry_fill_time"],
                utc=True,
            ).dt.year
        )

        trades["exit_year"] = (
            pd.to_datetime(
                trades["exit_fill_time"],
                utc=True,
            ).dt.year
        )

    for year, group in (
        performance_bars.groupby(
            performance_bars.index.year
        )
    ):
        returns = group[
            "strategy_return"
        ].dropna()

        total_return = float(
            (1.0 + returns).prod()
            - 1.0
        )

        if trades.empty:
            entered_trades = 0
            exited_trades = 0
            yearly_trade_pnl = 0.0
        else:
            entered_trades = int(
                (trades["entry_year"] == year)
                .sum()
            )

            exited_subset = trades.loc[
                trades["exit_year"] == year
            ]

            exited_trades = int(
                len(exited_subset)
            )

            yearly_trade_pnl = float(
                exited_subset[
                    "net_pnl"
                ].sum()
            )

        rows.append(
            {
                "year": int(year),

                "n_bars": int(
                    len(group)
                ),

                "start_equity": float(
                    group[
                        "mtm_equity"
                    ].iloc[0]
                ),

                "end_equity": float(
                    group[
                        "mtm_equity"
                    ].iloc[-1]
                ),

                "total_return": (
                    total_return
                ),

                "annualized_sharpe": (
                    calculate_period_sharpe(
                        returns,
                        periods_per_year,
                        min_return_observations,
                    )
                ),

                "max_drawdown": (
                    calculate_period_drawdown(
                        returns
                    )
                ),

                "exposure": float(
                    group[
                        "exposure_flag"
                    ].mean()
                ),

                "long_exposure": float(
                    group[
                        "long_exposure_flag"
                    ].mean()
                ),

                "short_exposure": float(
                    group[
                        "short_exposure_flag"
                    ].mean()
                ),

                "entered_trades": (
                    entered_trades
                ),

                "exited_trades": (
                    exited_trades
                ),

                "realized_trade_pnl": (
                    yearly_trade_pnl
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .set_index("year")
    )

In [130]:
def build_exit_reason_summary(
    trades: pd.DataFrame,
) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame(
            columns=[
                "n_trades",
                "total_net_pnl",
                "mean_net_pnl",
                "win_rate",
                "profit_factor",
            ]
        )

    rows = []

    for exit_reason, group in trades.groupby(
        "exit_reason",
        dropna=False,
        observed=True,
    ):
        pnl = group["net_pnl"]

        rows.append(
            {
                "exit_reason": exit_reason,

                "n_trades": int(
                    len(group)
                ),

                "total_net_pnl": float(
                    pnl.sum()
                ),

                "mean_net_pnl": float(
                    pnl.mean()
                ),

                "median_net_pnl": float(
                    pnl.median()
                ),

                "win_rate": float(
                    (pnl > 0).mean()
                ),

                "profit_factor": (
                    calculate_profit_factor(
                        pnl
                    )
                ),

                "average_holding_bars": (
                    float(
                        group[
                            "holding_bars"
                        ].mean()
                    )
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .set_index("exit_reason")
        .sort_values(
            "n_trades",
            ascending=False,
        )
    )

In [131]:
def find_entry_regime_columns(
    trades: pd.DataFrame,
) -> list[str]:
    return [
        col
        for col in trades.columns
        if (
            col.startswith("entry_signal__")
            and col.endswith(
                "_confirmed_state"
            )
        )
    ]

In [132]:
def clean_regime_column_name(
    column: str,
) -> str:
    result = column

    if result.startswith(
        "entry_signal__"
    ):
        result = result.removeprefix(
            "entry_signal__"
        )

    if result.endswith(
        "_confirmed_state"
    ):
        result = result.removesuffix(
            "_confirmed_state"
        )

    return result

In [133]:
def build_trade_group_summary(
    trades: pd.DataFrame,
    group_column: str,
) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame()

    rows = []

    total_pnl = float(
        trades["net_pnl"].sum()
    )

    for group_value, group in trades.groupby(
        group_column,
        dropna=False,
        observed=True,
    ):
        pnl = group["net_pnl"]

        if pd.isna(group_value):
            group_value = "<missing>"

        rows.append(
            {
                "group": group_value,

                "n_trades": int(
                    len(group)
                ),

                "long_trades": int(
                    (group["side"] == "long")
                    .sum()
                ),

                "short_trades": int(
                    (group["side"] == "short")
                    .sum()
                ),

                "total_net_pnl": float(
                    pnl.sum()
                ),

                "pnl_share": (
                    float(pnl.sum())
                    / total_pnl
                    if total_pnl != 0
                    else np.nan
                ),

                "mean_net_pnl": float(
                    pnl.mean()
                ),

                "median_net_pnl": float(
                    pnl.median()
                ),

                "mean_trade_return": float(
                    group[
                        "net_return_on_entry"
                    ].mean()
                ),

                "median_trade_return": float(
                    group[
                        "net_return_on_entry"
                    ].median()
                ),

                "win_rate": float(
                    (pnl > 0).mean()
                ),

                "profit_factor": (
                    calculate_profit_factor(
                        pnl
                    )
                ),

                "payoff_ratio": (
                    calculate_payoff_ratio(
                        pnl
                    )
                ),

                "average_holding_bars": (
                    float(
                        group[
                            "holding_bars"
                        ].mean()
                    )
                ),

                "average_mfe_return": (
                    float(
                        group[
                            "mfe_return_on_entry"
                        ].mean()
                    )
                ),

                "average_mae_return": (
                    float(
                        group[
                            "mae_return_on_entry"
                        ].mean()
                    )
                ),
            }
        )

    return (
        pd.DataFrame(rows)
        .set_index("group")
        .sort_values(
            "n_trades",
            ascending=False,
        )
    )

In [134]:
def build_regime_attribution(
    trades: pd.DataFrame,
) -> dict[str, pd.DataFrame]:
    result: dict[str, pd.DataFrame] = {}

    for column in find_entry_regime_columns(
        trades
    ):
        regime_name = (
            clean_regime_column_name(
                column
            )
        )

        result[regime_name] = (
            build_trade_group_summary(
                trades,
                column,
            )
        )

    return result

In [135]:
def analyze_execution_result(
    result: ExecutionResult,
    cfg: PerformanceConfig = PerformanceConfig(),
) -> PerformanceReport:
    performance_bars, periods_per_year = (
        prepare_performance_bars(
            result,
            cfg,
        )
    )

    drawdown_table, drawdown_metrics = (
        calculate_drawdown_diagnostics(
            performance_bars,
            result.execution_timeframe,
        )
    )

    # ใส่ drawdown duration กลับเข้า bar table
    performance_bars[
        "drawdown_duration_bars"
    ] = drawdown_table[
        "drawdown_duration_bars"
    ]

    performance_bars[
        "drawdown_duration"
    ] = drawdown_table[
        "drawdown_duration"
    ]

    return_risk_metrics = (
        calculate_return_risk_metrics(
            performance_bars,

            initial_capital=(
                result.config.initial_capital
            ),

            periods_per_year=(
                periods_per_year
            ),

            annualization_days=(
                cfg.annualization_days
            ),

            min_return_observations=(
                cfg.min_return_observations
            ),
        )
    )

    trade_summary = (
        summarize_trade_dataframe(
            result.trades
        )
    )

    duration_years = (
        return_risk_metrics[
            "duration_years"
        ]
    )

    execution_metrics = (
        calculate_execution_metrics(
            result,
            performance_bars,
            duration_years,
        )
    )

    summary_data = {
        "strategy_id": result.strategy_id,
        "execution_timeframe": (
            result.execution_timeframe
        ),

        "periods_per_year": (
            periods_per_year
        ),

        **return_risk_metrics,
        **drawdown_metrics,
        **execution_metrics,

        "trade_win_rate": (
            trade_summary["win_rate"]
        ),

        "trade_profit_factor": (
            trade_summary["profit_factor"]
        ),

        "mean_trade_return": (
            trade_summary[
                "mean_trade_return"
            ]
        ),

        "median_trade_return": (
            trade_summary[
                "median_trade_return"
            ]
        ),

        "average_holding_bars": (
            trade_summary[
                "average_holding_bars"
            ]
        ),

        "average_mfe_return": (
            trade_summary[
                "average_mfe_return"
            ]
        ),

        "average_mae_return": (
            trade_summary[
                "average_mae_return"
            ]
        ),
    }

    summary = pd.Series(
        summary_data
    )

    side_summary = build_side_summary(
        result.trades
    )

    yearly_summary = build_yearly_summary(
        result,
        performance_bars,
        periods_per_year,
        cfg.min_return_observations,
    )

    exit_reason_summary = (
        build_exit_reason_summary(
            result.trades
        )
    )

    regime_attribution = (
        build_regime_attribution(
            result.trades
        )
    )

    drawdown_output = (
        performance_bars[
            [
                "mtm_equity",
                "realized_equity",
                "equity_peak",
                "drawdown",
                "underwater",
                "drawdown_duration_bars",
                "drawdown_duration",
            ]
        ].copy()
    )

    return PerformanceReport(
        strategy_id=result.strategy_id,
        execution_timeframe=(
            result.execution_timeframe
        ),

        summary=summary,
        trade_summary=trade_summary,

        side_summary=side_summary,
        yearly_summary=yearly_summary,
        exit_reason_summary=(
            exit_reason_summary
        ),

        regime_attribution=(
            regime_attribution
        ),

        drawdown=drawdown_output,
    )

In [136]:
def validate_performance_report(
    execution_result: ExecutionResult,
    report: PerformanceReport,
) -> None:
    summary = report.summary

    required_summary = {
        "initial_capital",
        "final_mtm_equity",
        "total_return",
        "cagr",
        "annualized_sharpe",
        "max_drawdown",

        "exposure",
        "long_exposure",
        "short_exposure",

        "turnover",
        "n_closed_trades",
    }

    missing = required_summary.difference(
        summary.index
    )

    if missing:
        raise AssertionError(
            f"Missing performance metrics: "
            f"{sorted(missing)}"
        )

    expected_final_equity = float(
        execution_result.bars[
            "mtm_equity"
        ].iloc[-1]
    )

    if not np.isclose(
        float(summary["final_mtm_equity"]),
        expected_final_equity,
        equal_nan=True,
    ):
        raise AssertionError(
            "Final equity does not match "
            "ExecutionResult."
        )

    expected_total_return = (
        expected_final_equity
        / execution_result.config.initial_capital
        - 1.0
    )

    if not np.isclose(
        float(summary["total_return"]),
        expected_total_return,
        equal_nan=True,
    ):
        raise AssertionError(
            "Total return is inconsistent."
        )

    if float(summary["max_drawdown"]) > 0:
        raise AssertionError(
            "Maximum drawdown cannot be positive."
        )

    exposure_sum = (
        float(summary["long_exposure"])
        + float(summary["short_exposure"])
    )

    if not np.isclose(
        float(summary["exposure"]),
        exposure_sum,
        atol=1e-12,
        equal_nan=True,
    ):
        raise AssertionError(
            "Long + short exposure != exposure."
        )

    if not execution_result.trades.empty:
        report_trade_count = int(
            report.trade_summary[
                "n_trades"
            ]
        )

        if report_trade_count != len(
            execution_result.trades
        ):
            raise AssertionError(
                "Trade count does not match."
            )

    # เมื่อปิดทุก Position แล้ว
    # final realized equity ต้องเท่ากับ
    # initial + sum(net PnL)
    if execution_result.open_position is None:
        expected_realized = (
            execution_result.config.initial_capital
            + execution_result.trades[
                "net_pnl"
            ].sum()
        )

        actual_realized = float(
            execution_result.bars[
                "realized_equity"
            ].iloc[-1]
        )

        if not np.isclose(
            actual_realized,
            expected_realized,
            atol=1e-8,
            equal_nan=True,
        ):
            raise AssertionError(
                "Realized equity does not equal "
                "initial capital + closed trade PnL."
            )

In [137]:
def compare_performance_reports(
    reports: dict[str, PerformanceReport],
) -> pd.DataFrame:
    rows = []

    for label, report in reports.items():
        row = report.summary.copy()
        row["label"] = label

        rows.append(row)

    result = pd.DataFrame(rows)

    return result.set_index("label")

In [138]:
def compare_against_baseline(
    comparison: pd.DataFrame,
    baseline_label: str,
) -> pd.DataFrame:
    if baseline_label not in comparison.index:
        raise KeyError(
            f"Baseline not found: "
            f"{baseline_label}"
        )

    baseline = comparison.loc[
        baseline_label
    ]

    metrics = [
        "total_return",
        "cagr",
        "annualized_sharpe",
        "annualized_sortino",
        "max_drawdown",
        "calmar_ratio",

        "exposure",
        "n_closed_trades",
        "trade_win_rate",
        "trade_profit_factor",
        "mean_trade_return",

        "turnover",
        "total_fees",
        "total_slippage_cost",
    ]

    rows = []

    for label, row in comparison.iterrows():
        output = {
            "label": label,
        }

        for metric in metrics:
            if metric not in comparison.columns:
                continue

            value = row[metric]
            baseline_value = baseline[metric]

            output[metric] = value

            output[
                f"{metric}_delta"
            ] = (
                value - baseline_value
                if (
                    pd.notna(value)
                    and pd.notna(
                        baseline_value
                    )
                )
                else np.nan
            )

        rows.append(output)

    return (
        pd.DataFrame(rows)
        .set_index("label")
    )

# Visualization Engine

In [139]:
# Portfolio
# ├── MTM Equity
# ├── Realized Equity
# ├── Drawdown
# └── Position / Exposure

# Market
# ├── ราคา + Keltner
# ├── Regime background
# └── Entry / Exit ทั้งหมด

# Individual Trade
# ├── ก่อนเข้า
# ├── ระหว่างถือ
# ├── หลังออก
# ├── Entry / Exit
# ├── MFE / MAE
# ├── Unrealized PnL path
# └── Equity ขณะเกิด Trade

In [140]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle

In [141]:
@dataclass(frozen=True)
class VisualizationConfig:
    pre_trade_bars: int = 30
    post_trade_bars: int = 30

    candle_width: float = 0.65
    figure_dpi: int = 120

    show_keltner: bool = True
    show_regime_background: bool = True
    show_mfe_mae: bool = True

    # เช่น imom_imacd_4h_confirmed_state
    regime_column: Optional[str] = None

In [142]:
DEFAULT_REGIME_COLOR_MAP = {
    # Strong (+)
    "plus_lime":   "#d8f3c8",   # pastel lime
    "plus_green":  "#dcefd8",   # pale mint
    "plus_red":    "#f6d3d3",   # soft pink
    "plus_orange": "#f8ead0",   # light cream

    # Flat
    "flat_lime":   "#e7f8df",
    "flat_green":  "#e5f3e5",
    "flat_red":    "#f9e3e3",
    "flat_orange": "#fbf0dc",

    # Minus (-)
    "minus_lime":  "#cfecc2",
    "minus_green": "#d6e8d5",
    "minus_red":   "#efc3c3",
    "minus_orange":"#f5dfb9",
}

In [143]:
def infer_regime_color(
    state: object,
) -> str:
    if state is None or pd.isna(state):
        return "#eeeeee"

    state = str(state)

    if state in DEFAULT_REGIME_COLOR_MAP:
        return DEFAULT_REGIME_COLOR_MAP[state]

    suffix = state.rsplit("_", maxsplit=1)[-1]

    suffix_map = {
        "lime": "#c8f7b8",
        "green": "#b8dbc1",
        "red": "#f2b5b5",
        "orange": "#f8d5a4",
    }

    return suffix_map.get(
        suffix,
        "#dddddd",
    )

In [144]:
def deduplicate_legend(
    ax: Axes,
) -> None:
    handles, labels = (
        ax.get_legend_handles_labels()
    )

    unique = {}

    for handle, label in zip(
        handles,
        labels,
    ):
        if not label:
            continue

        if label not in unique:
            unique[label] = handle

    if unique:
        ax.legend(
            unique.values(),
            unique.keys(),
            loc="best",
        )

In [145]:
def draw_candlesticks(
    ax: Axes,
    bars: pd.DataFrame,
    *,
    candle_width: float = 0.65,
    up_color: str = "#2ca02c",
    down_color: str = "#d62728",
    flat_color: str = "#7f7f7f",
    alpha: float = 0.9,
) -> np.ndarray:
    required = {
        "open",
        "high",
        "low",
        "close",
    }

    missing = required.difference(
        bars.columns
    )

    if missing:
        raise ValueError(
            f"Missing OHLC columns: "
            f"{sorted(missing)}"
        )

    x = np.arange(
        len(bars),
        dtype=float,
    )

    opens = bars["open"].to_numpy(
        dtype=float
    )

    highs = bars["high"].to_numpy(
        dtype=float
    )

    lows = bars["low"].to_numpy(
        dtype=float
    )

    closes = bars["close"].to_numpy(
        dtype=float
    )

    for i in range(len(bars)):
        open_price = opens[i]
        high_price = highs[i]
        low_price = lows[i]
        close_price = closes[i]

        if not np.isfinite(
            [
                open_price,
                high_price,
                low_price,
                close_price,
            ]
        ).all():
            continue

        if close_price > open_price:
            color = up_color

        elif close_price < open_price:
            color = down_color

        else:
            color = flat_color

        ax.vlines(
            x[i],
            low_price,
            high_price,
            color=color,
            linewidth=1.0,
            alpha=alpha,
        )

        body_low = min(
            open_price,
            close_price,
        )

        body_height = abs(
            close_price - open_price
        )

        if body_height == 0:
            ax.hlines(
                open_price,
                x[i] - candle_width / 2,
                x[i] + candle_width / 2,
                color=color,
                linewidth=1.4,
                alpha=alpha,
            )

        else:
            rectangle = Rectangle(
                (
                    x[i] - candle_width / 2,
                    body_low,
                ),
                candle_width,
                body_height,
                facecolor=color,
                edgecolor=color,
                alpha=alpha,
            )

            ax.add_patch(rectangle)

    return x

In [146]:
def set_timestamp_axis_labels(
    ax: Axes,
    index: pd.DatetimeIndex,
    *,
    max_ticks: int = 10,
) -> None:
    n = len(index)

    if n == 0:
        return

    n_ticks = min(
        max_ticks,
        n,
    )

    positions = np.linspace(
        0,
        n - 1,
        num=n_ticks,
        dtype=int,
    )

    positions = np.unique(positions)

    labels = [
        pd.Timestamp(index[position]).strftime(
            "%Y-%m-%d\n%H:%M"
        )
        for position in positions
    ]

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)

In [147]:
def extract_regime_segments_from_bars(
    bars: pd.DataFrame,
    regime_column: str,
) -> pd.DataFrame:
    if regime_column not in bars.columns:
        raise KeyError(
            f"Regime column not found: "
            f"{regime_column}"
        )

    states = bars[
        regime_column
    ].astype("string")

    state_change = (
        states.ne(states.shift(1))
        | states.isna().ne(
            states.shift(1).isna()
        )
    )

    segment_id = (
        state_change.astype(int).cumsum()
    )

    rows = []

    for _, group in bars.groupby(
        segment_id,
        sort=False,
    ):
        start_position = (
            bars.index.get_loc(
                group.index[0]
            )
        )

        end_position = (
            bars.index.get_loc(
                group.index[-1]
            )
        )

        state = group[
            regime_column
        ].iloc[0]

        rows.append(
            {
                "start_position": (
                    start_position
                ),
                "end_position": (
                    end_position
                ),
                "state": state,
                "start_time": (
                    group.index[0]
                ),
                "end_time": (
                    group.index[-1]
                ),
            }
        )

    return pd.DataFrame(rows)

In [148]:
def shade_regime_background(
    ax: Axes,
    bars: pd.DataFrame,
    regime_column: str,
    *,
    alpha: float = 0.6,
) -> None:
    segments = (
        extract_regime_segments_from_bars(
            bars,
            regime_column,
        )
    )

    for row in segments.itertuples():
        color = infer_regime_color(
            row.state
        )

        ax.axvspan(
            row.start_position - 0.5,
            row.end_position + 0.5,
            facecolor=color,
            alpha=alpha,
            linewidth=0,
        )

In [149]:
def plot_equity_and_drawdown(
    execution_result: ExecutionResult,
    performance_report: Optional[
        PerformanceReport
    ] = None,
    *,
    title: Optional[str] = None,
    figsize: tuple[float, float] = (
        15,
        9,
    ),
) -> tuple[Figure, np.ndarray]:
    bars = execution_result.bars

    required = {
        "mtm_equity",
        "realized_equity",
        "position_sign",
    }

    missing = required.difference(
        bars.columns
    )

    if missing:
        raise ValueError(
            f"Missing equity columns: "
            f"{sorted(missing)}"
        )

    if performance_report is not None:
        drawdown = (
            performance_report
            .drawdown["drawdown"]
        )
    else:
        equity_peak = (
            bars["mtm_equity"].cummax()
        )

        drawdown = (
            bars["mtm_equity"]
            / equity_peak
            - 1.0
        )

    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=figsize,
        sharex=True,
        gridspec_kw={
            "height_ratios": [
                3.0,
                1.5,
                1.0,
            ]
        },
    )

    equity_ax = axes[0]
    drawdown_ax = axes[1]
    position_ax = axes[2]

    equity_ax.plot(
        bars.index,
        bars["mtm_equity"],
        label="Mark-to-market equity",
        linewidth=1.3,
    )

    equity_ax.plot(
        bars.index,
        bars["realized_equity"],
        label="Realized equity",
        linewidth=1.1,
        alpha=0.85,
    )

    equity_ax.set_ylabel("Equity")
    equity_ax.grid(
        True,
        alpha=0.25,
    )
    equity_ax.legend(loc="best")

    drawdown_ax.fill_between(
        bars.index,
        drawdown.to_numpy(
            dtype=float
        ),
        0.0,
        alpha=0.4,
    )

    drawdown_ax.plot(
        bars.index,
        drawdown,
        linewidth=0.8,
    )

    drawdown_ax.set_ylabel("Drawdown")
    drawdown_ax.grid(
        True,
        alpha=0.25,
    )

    position_ax.step(
        bars.index,
        bars["position_sign"],
        where="post",
        linewidth=1.0,
    )

    position_ax.axhline(
        0,
        linewidth=0.8,
        alpha=0.6,
    )

    position_ax.set_yticks(
        [-1, 0, 1]
    )

    position_ax.set_yticklabels(
        ["Short", "Flat", "Long"]
    )

    position_ax.set_ylabel("Position")
    position_ax.set_xlabel("Time")
    position_ax.grid(
        True,
        alpha=0.25,
    )

    if title is None:
        title = (
            execution_result.strategy_id
        )

    fig.suptitle(title)

    fig.tight_layout()

    return fig, axes

In [150]:
# fig, axes = plot_equity_and_drawdown(
#     breakout_4h_execution,
#     breakout_4h_report,
#     title="Keltner Breakout + 4h Regime",
# )

# plt.show()

In [151]:
def plot_strategy_equity_comparison(
    execution_results: dict[
        str,
        ExecutionResult,
    ],
    *,
    normalize: bool = True,
    equity_type: str = "mtm_equity",
    figsize: tuple[float, float] = (
        15,
        6,
    ),
) -> tuple[Figure, Axes]:
    fig, ax = plt.subplots(
        figsize=figsize
    )

    for label, result in (
        execution_results.items()
    ):
        equity = result.bars[
            equity_type
        ].astype(float)

        if normalize:
            initial = (
                result.config.initial_capital
            )

            plot_value = equity / initial
            ylabel = "Growth of 1.0"

        else:
            plot_value = equity
            ylabel = "Equity"

        ax.plot(
            result.bars.index,
            plot_value,
            label=label,
            linewidth=1.2,
        )

    ax.set_title(
        f"Strategy comparison: {equity_type}"
    )

    ax.set_ylabel(ylabel)
    ax.set_xlabel("Time")

    ax.grid(
        True,
        alpha=0.25,
    )

    ax.legend(loc="best")

    fig.tight_layout()

    return fig, ax

In [152]:
# fig, ax = plot_strategy_equity_comparison(
#     {
#         "Breakout baseline": (
#             breakout_baseline_execution
#         ),
#         "Breakout + 4h": (
#             breakout_4h_execution
#         ),
#         # "Mean reversion baseline": (
#         #     mean_reversion_baseline_execution
#         # ),
#         # "Mean reversion + 4h flat": (
#         #     mean_reversion_4h_flat_execution
#         # ),
#     }
# )

# plt.show()

In [153]:
@dataclass
class TradePlotData:
    trade: pd.Series
    bars: pd.DataFrame

    trade_id: int

    entry_position: int
    exit_position: int

    entry_local_position: int
    exit_local_position: int

In [154]:
def get_trade_plot_data(
    execution_result: ExecutionResult,
    trade_id: int,
    *,
    pre_bars: int = 30,
    post_bars: int = 30,
) -> TradePlotData:
    trades = execution_result.trades
    bars = execution_result.bars

    if trades.empty:
        raise ValueError(
            "Execution result has no closed trades."
        )

    matches = trades.loc[
        trades["trade_id"] == trade_id
    ]

    if matches.empty:
        raise KeyError(
            f"Trade ID not found: {trade_id}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"Duplicated trade ID: {trade_id}"
        )

    trade = matches.iloc[0]

    entry_bar_end = pd.Timestamp(
        trade["entry_fill_bar_end"]
    )

    exit_bar_end = pd.Timestamp(
        trade["exit_fill_bar_end"]
    )

    if entry_bar_end not in bars.index:
        raise KeyError(
            "Entry fill bar is not present "
            f"in execution bars: {entry_bar_end}"
        )

    if exit_bar_end not in bars.index:
        raise KeyError(
            "Exit fill bar is not present "
            f"in execution bars: {exit_bar_end}"
        )

    entry_position = int(
        bars.index.get_loc(entry_bar_end)
    )

    exit_position = int(
        bars.index.get_loc(exit_bar_end)
    )

    window_start = max(
        0,
        entry_position - pre_bars,
    )

    window_end = min(
        len(bars) - 1,
        exit_position + post_bars,
    )

    trade_bars = bars.iloc[
        window_start:
        window_end + 1
    ].copy()

    entry_local_position = (
        entry_position - window_start
    )

    exit_local_position = (
        exit_position - window_start
    )

    return TradePlotData(
        trade=trade,
        bars=trade_bars,
        trade_id=trade_id,

        entry_position=entry_position,
        exit_position=exit_position,

        entry_local_position=(
            entry_local_position
        ),
        exit_local_position=(
            exit_local_position
        ),
    )

In [155]:
def calculate_trade_pnl_path(
    trade_plot_data: TradePlotData,
) -> pd.Series:
    trade = trade_plot_data.trade
    bars = trade_plot_data.bars

    side = str(trade["side"])

    sign = (
        1.0
        if side == "long"
        else -1.0
    )

    quantity = float(
        trade["quantity"]
    )

    entry_price = float(
        trade["entry_fill_price"]
    )

    entry_bar_end = pd.Timestamp(
        trade["entry_fill_bar_end"]
    )

    exit_bar_end = pd.Timestamp(
        trade["exit_fill_bar_end"]
    )

    exit_fill_time = pd.Timestamp(
        trade["exit_fill_time"]
    )

    # end_of_data อาจปิดที่ close ของ exit bar
    exit_at_bar_close = (
        exit_fill_time == exit_bar_end
    )

    if exit_at_bar_close:
        active_mask = (
            (bars.index >= entry_bar_end)
            & (bars.index <= exit_bar_end)
        )
    else:
        active_mask = (
            (bars.index >= entry_bar_end)
            & (bars.index < exit_bar_end)
        )

    pnl = pd.Series(
        np.nan,
        index=bars.index,
        dtype=float,
    )

    pnl.loc[active_mask] = (
        sign
        * quantity
        * (
            bars.loc[
                active_mask,
                "close",
            ]
            - entry_price
        )
    )

    # จุดสุดท้ายคือ PnL ตอนออกจริง
    pnl.loc[exit_bar_end] = float(
        trade["gross_pnl_after_slippage"]
    )

    return pnl

In [156]:
def calculate_trade_excursion_prices(
    trade: pd.Series,
) -> tuple[
    Optional[float],
    Optional[float],
]:
    quantity = float(
        trade["quantity"]
    )

    if quantity <= 0:
        return None, None

    entry_price = float(
        trade["entry_fill_price"]
    )

    mfe_pnl = trade.get(
        "mfe_pnl",
        np.nan,
    )

    mae_pnl = trade.get(
        "mae_pnl",
        np.nan,
    )

    if trade["side"] == "long":
        mfe_price = (
            entry_price
            + float(mfe_pnl) / quantity
            if pd.notna(mfe_pnl)
            else None
        )

        mae_price = (
            entry_price
            + float(mae_pnl) / quantity
            if pd.notna(mae_pnl)
            else None
        )

    else:
        mfe_price = (
            entry_price
            - float(mfe_pnl) / quantity
            if pd.notna(mfe_pnl)
            else None
        )

        mae_price = (
            entry_price
            - float(mae_pnl) / quantity
            if pd.notna(mae_pnl)
            else None
        )

    return mfe_price, mae_price

In [157]:
def plot_trade(
    execution_result: ExecutionResult,
    trade_id: int,
    *,
    cfg: VisualizationConfig = (
        VisualizationConfig()
    ),
    title: Optional[str] = None,
) -> tuple[Figure, np.ndarray]:
    trade_data = get_trade_plot_data(
        execution_result,
        trade_id,

        pre_bars=cfg.pre_trade_bars,
        post_bars=cfg.post_trade_bars,
    )

    trade = trade_data.trade
    bars = trade_data.bars

    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(16, 11),
        sharex=True,
        gridspec_kw={
            "height_ratios": [
                3.5,
                1.4,
                1.4,
            ]
        },
        dpi=cfg.figure_dpi,
    )

    price_ax = axes[0]
    pnl_ax = axes[1]
    equity_ax = axes[2]

    # -------------------------
    # Regime background
    # -------------------------

    if (
        cfg.show_regime_background
        and cfg.regime_column is not None
        and cfg.regime_column
        in bars.columns
    ):
        shade_regime_background(
            price_ax,
            bars,
            cfg.regime_column,
        )

    # -------------------------
    # Candlesticks
    # -------------------------

    x = draw_candlesticks(
        price_ax,
        bars,
        candle_width=(
            cfg.candle_width
        ),
    )

    # -------------------------
    # Keltner Channel
    # -------------------------

    if cfg.show_keltner:
        if "kc_basis" in bars.columns:
            price_ax.plot(
                x,
                bars["kc_basis"],
                label="KC basis",
                linewidth=1.0,
            )

        if "kc_upper" in bars.columns:
            price_ax.plot(
                x,
                bars["kc_upper"],
                label="KC upper",
                linewidth=0.9,
                alpha=0.8,
            )

        if "kc_lower" in bars.columns:
            price_ax.plot(
                x,
                bars["kc_lower"],
                label="KC lower",
                linewidth=0.9,
                alpha=0.8,
            )

    entry_x = (
        trade_data.entry_local_position
    )

    exit_x = (
        trade_data.exit_local_position
    )

    entry_price = float(
        trade["entry_fill_price"]
    )

    exit_price = float(
        trade["exit_fill_price"]
    )

    side = str(trade["side"])

    if side == "long":
        entry_marker = "^"
        entry_label = "Long entry"

    else:
        entry_marker = "v"
        entry_label = "Short entry"

    price_ax.scatter(
        entry_x,
        entry_price,
        marker=entry_marker,
        s=120,
        zorder=10,
        label=entry_label,
    )

    price_ax.scatter(
        exit_x,
        exit_price,
        marker="X",
        s=110,
        zorder=10,
        label="Exit",
    )

    price_ax.axvline(
        entry_x,
        linestyle="--",
        linewidth=0.9,
        alpha=0.7,
    )

    price_ax.axvline(
        exit_x,
        linestyle="--",
        linewidth=0.9,
        alpha=0.7,
    )

    # -------------------------
    # MFE / MAE
    # -------------------------

    if cfg.show_mfe_mae:
        mfe_price, mae_price = (
            calculate_trade_excursion_prices(
                trade
            )
        )

        mfe_time = trade.get(
            "mfe_time",
            pd.NaT,
        )

        mae_time = trade.get(
            "mae_time",
            pd.NaT,
        )

        if (
            pd.notna(mfe_time)
            and pd.Timestamp(mfe_time)
            in bars.index
            and mfe_price is not None
        ):
            mfe_x = bars.index.get_loc(
                pd.Timestamp(mfe_time)
            )

            price_ax.scatter(
                mfe_x,
                mfe_price,
                marker="*",
                s=130,
                label="MFE",
                zorder=10,
            )

        if (
            pd.notna(mae_time)
            and pd.Timestamp(mae_time)
            in bars.index
            and mae_price is not None
        ):
            mae_x = bars.index.get_loc(
                pd.Timestamp(mae_time)
            )

            price_ax.scatter(
                mae_x,
                mae_price,
                marker="P",
                s=100,
                label="MAE",
                zorder=10,
            )

    price_ax.set_ylabel("Price")

    price_ax.grid(
        True,
        alpha=0.22,
    )

    deduplicate_legend(
        price_ax
    )

    # -------------------------
    # Trade PnL Path
    # -------------------------

    trade_pnl_path = (
        calculate_trade_pnl_path(
            trade_data
        )
    )

    pnl_ax.plot(
        x,
        trade_pnl_path.to_numpy(
            dtype=float
        ),
        label="Trade PnL",
        linewidth=1.2,
    )

    pnl_ax.axhline(
        0,
        linewidth=0.8,
        alpha=0.7,
    )

    pnl_ax.axvline(
        entry_x,
        linestyle="--",
        linewidth=0.8,
        alpha=0.6,
    )

    pnl_ax.axvline(
        exit_x,
        linestyle="--",
        linewidth=0.8,
        alpha=0.6,
    )

    pnl_ax.scatter(
        exit_x,
        float(trade["net_pnl"]),
        marker="X",
        s=80,
        label="Net PnL",
        zorder=10,
    )

    pnl_ax.set_ylabel("PnL")

    pnl_ax.grid(
        True,
        alpha=0.22,
    )

    deduplicate_legend(
        pnl_ax
    )

    # -------------------------
    # Portfolio Equity
    # -------------------------

    if "mtm_equity" in bars.columns:
        equity_ax.plot(
            x,
            bars["mtm_equity"],
            label="MTM equity",
            linewidth=1.1,
        )

    if "realized_equity" in bars.columns:
        equity_ax.plot(
            x,
            bars["realized_equity"],
            label="Realized equity",
            linewidth=1.0,
            alpha=0.85,
        )

    equity_ax.axvline(
        entry_x,
        linestyle="--",
        linewidth=0.8,
        alpha=0.6,
    )

    equity_ax.axvline(
        exit_x,
        linestyle="--",
        linewidth=0.8,
        alpha=0.6,
    )

    equity_ax.set_ylabel("Equity")
    equity_ax.set_xlabel("Time")

    equity_ax.grid(
        True,
        alpha=0.22,
    )

    deduplicate_legend(
        equity_ax
    )

    set_timestamp_axis_labels(
        equity_ax,
        bars.index,
        max_ticks=10,
    )

    # -------------------------
    # Title
    # -------------------------

    if title is None:
        regime_text = ""

        if (
            cfg.regime_column is not None
            and (
                f"entry_signal__"
                f"{cfg.regime_column}"
            )
            in trade.index
        ):
            entry_regime = trade[
                f"entry_signal__"
                f"{cfg.regime_column}"
            ]

            regime_text = (
                f" | Entry regime: "
                f"{entry_regime}"
            )

        title = (
            f"Trade {trade_id} | "
            f"{side.upper()} | "
            f"Net PnL: "
            f"{float(trade['net_pnl']):,.2f} | "
            f"Return: "
            f"{float(trade['net_return_on_entry']):.2%} | "
            f"Holding: "
            f"{int(trade['holding_bars'])} bars"
            f"{regime_text}"
        )

    fig.suptitle(title)

    fig.tight_layout()

    return fig, axes

In [158]:
# trade_cfg = VisualizationConfig(
#     pre_trade_bars=40,
#     post_trade_bars=40,

#     regime_column=(
#         "imom_imacd_4h_confirmed_state"
#     ),

#     show_keltner=True,
#     show_regime_background=True,
#     show_mfe_mae=True,
# )

In [159]:
# fig, axes = plot_trade(
#     breakout_4h_execution,
#     trade_id=1,
#     cfg=trade_cfg,
# )

# plt.show()

In [160]:
def get_extreme_trade_ids(
    execution_result: ExecutionResult,
    *,
    n: int = 5,
) -> dict[str, list[int]]:
    trades = execution_result.trades

    if trades.empty:
        return {
            "best": [],
            "worst": [],
            "highest_mfe": [],
            "lowest_mae": [],
        }

    best = (
        trades.nlargest(
            n,
            "net_pnl",
        )["trade_id"]
        .astype(int)
        .tolist()
    )

    worst = (
        trades.nsmallest(
            n,
            "net_pnl",
        )["trade_id"]
        .astype(int)
        .tolist()
    )

    highest_mfe = (
        trades.nlargest(
            n,
            "mfe_return_on_entry",
        )["trade_id"]
        .astype(int)
        .tolist()
    )

    lowest_mae = (
        trades.nsmallest(
            n,
            "mae_return_on_entry",
        )["trade_id"]
        .astype(int)
        .tolist()
    )

    return {
        "best": best,
        "worst": worst,
        "highest_mfe": (
            highest_mfe
        ),
        "lowest_mae": lowest_mae,
    }

In [161]:
# extreme_trades = get_extreme_trade_ids(
#     breakout_4h_execution,
#     n=5,
# )

# print(extreme_trades)

In [162]:
# fig, axes = plot_trade(
#     breakout_4h_execution,
#     trade_id=extreme_trades["worst"][0],
#     cfg=trade_cfg,
# )

# plt.show()

In [163]:
def select_trade_ids_by_regime(
    execution_result: ExecutionResult,
    *,
    regime_column: str,
    states: list[str],
    side: Optional[str] = None,
    profitable: Optional[bool] = None,
    limit: Optional[int] = None,
) -> list[int]:
    trades = execution_result.trades

    trade_regime_column = (
        f"entry_signal__{regime_column}"
    )

    if trade_regime_column not in trades.columns:
        raise KeyError(
            "Trade ledger does not contain "
            f"{trade_regime_column}"
        )

    mask = trades[
        trade_regime_column
    ].isin(states)

    if side is not None:
        mask &= (
            trades["side"] == side
        )

    if profitable is True:
        mask &= trades["net_pnl"] > 0

    elif profitable is False:
        mask &= trades["net_pnl"] <= 0

    selected = trades.loc[
        mask,
        "trade_id",
    ].astype(int)

    if limit is not None:
        selected = selected.head(limit)

    return selected.tolist()

In [164]:
# plus_lime_long_trades = (
#     select_trade_ids_by_regime(
#         breakout_baseline_execution,

#         regime_column=(
#             "imom_imacd_4h_confirmed_state"
#         ),

#         states=[
#             "plus_lime",
#         ],

#         side="long",
#         limit=10,
#     )
# )

# print(plus_lime_long_trades)

In [165]:
def export_trade_charts(
    execution_result: ExecutionResult,
    output_directory: str | Path,
    *,
    cfg: VisualizationConfig = (
        VisualizationConfig()
    ),
    trade_ids: Optional[list[int]] = None,
    max_trades: Optional[int] = None,
    image_format: str = "png",
) -> list[Path]:
    output_directory = Path(
        output_directory
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    if trade_ids is None:
        trade_ids = (
            execution_result.trades[
                "trade_id"
            ]
            .astype(int)
            .tolist()
        )

    if max_trades is not None:
        trade_ids = trade_ids[
            :max_trades
        ]

    output_paths = []

    for trade_id in trade_ids:
        fig, _ = plot_trade(
            execution_result,
            trade_id,
            cfg=cfg,
        )

        output_path = (
            output_directory
            / (
                f"trade_{trade_id:06d}."
                f"{image_format}"
            )
        )

        fig.savefig(
            output_path,
            bbox_inches="tight",
            dpi=cfg.figure_dpi,
        )

        plt.close(fig)

        output_paths.append(
            output_path
        )

    return output_paths

In [166]:
def plot_strategy_trades_on_price(
    execution_result: ExecutionResult,
    *,
    start: Optional[
        str | pd.Timestamp
    ] = None,
    end: Optional[
        str | pd.Timestamp
    ] = None,
    regime_column: Optional[str] = None,
    show_keltner: bool = True,
    figsize: tuple[float, float] = (
        17,
        7,
    ),
) -> tuple[Figure, Axes]:
    bars = execution_result.bars.copy()
    trades = execution_result.trades.copy()

    if start is not None:
        start = pd.Timestamp(
            start,
            tz="UTC",
        )

        bars = bars.loc[
            bars.index >= start
        ]

    if end is not None:
        end = pd.Timestamp(
            end,
            tz="UTC",
        )

        bars = bars.loc[
            bars.index <= end
        ]

    if bars.empty:
        raise ValueError(
            "No bars in requested period."
        )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    if (
        regime_column is not None
        and regime_column in bars.columns
    ):
        shade_regime_background(
            ax,
            bars,
            regime_column,
        )

    x = draw_candlesticks(
        ax,
        bars,
    )

    if show_keltner:
        for col, label in [
            ("kc_basis", "KC basis"),
            ("kc_upper", "KC upper"),
            ("kc_lower", "KC lower"),
        ]:
            if col in bars.columns:
                ax.plot(
                    x,
                    bars[col],
                    label=label,
                    linewidth=0.9,
                )

    position_map = {
        timestamp: position
        for position, timestamp
        in enumerate(bars.index)
    }

    for trade in trades.itertuples():
        entry_bar_end = pd.Timestamp(
            trade.entry_fill_bar_end
        )

        exit_bar_end = pd.Timestamp(
            trade.exit_fill_bar_end
        )

        if entry_bar_end in position_map:
            marker = (
                "^"
                if trade.side == "long"
                else "v"
            )

            ax.scatter(
                position_map[
                    entry_bar_end
                ],
                trade.entry_fill_price,
                marker=marker,
                s=45,
                zorder=10,
                label=(
                    f"{trade.side.title()} entry"
                ),
            )

        if exit_bar_end in position_map:
            ax.scatter(
                position_map[
                    exit_bar_end
                ],
                trade.exit_fill_price,
                marker="X",
                s=40,
                zorder=10,
                label="Exit",
            )

    ax.set_title(
        execution_result.strategy_id
    )

    ax.set_ylabel("Price")
    ax.set_xlabel("Time")

    ax.grid(
        True,
        alpha=0.22,
    )

    set_timestamp_axis_labels(
        ax,
        bars.index,
        max_ticks=12,
    )

    deduplicate_legend(ax)

    fig.tight_layout()

    return fig, ax

In [167]:
# fig, ax = plot_strategy_trades_on_price(
#     breakout_4h_execution,

#     start="2025-01-01",
#     end="2025-03-01",

#     regime_column=(
#         "imom_imacd_4h_confirmed_state"
#     ),
# )

# plt.show()

In [168]:
def plot_regime_attribution(
    performance_report: PerformanceReport,
    regime_name: str,
    *,
    metric: str = "mean_trade_return",
    min_trades: int = 1,
    figsize: tuple[float, float] = (
        12,
        6,
    ),
) -> tuple[Figure, Axes]:
    if (
        regime_name
        not in performance_report
        .regime_attribution
    ):
        raise KeyError(
            f"Regime attribution not found: "
            f"{regime_name}"
        )

    attribution = (
        performance_report
        .regime_attribution[
            regime_name
        ]
        .copy()
    )

    if metric not in attribution.columns:
        raise KeyError(
            f"Metric not found: {metric}"
        )

    attribution = attribution.loc[
        attribution["n_trades"]
        >= min_trades
    ]

    attribution = attribution.sort_values(
        metric,
        ascending=True,
    )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    ax.barh(
        attribution.index.astype(str),
        attribution[metric],
    )

    ax.axvline(
        0,
        linewidth=0.8,
        alpha=0.7,
    )

    ax.set_title(
        f"{regime_name}: {metric}"
    )

    ax.set_xlabel(metric)
    ax.set_ylabel("Entry regime")

    ax.grid(
        True,
        axis="x",
        alpha=0.25,
    )

    fig.tight_layout()

    return fig, ax

In [169]:
# fig, ax = plot_regime_attribution(
#     breakout_baseline_report,

#     regime_name="imom_imacd_4h",
#     metric="mean_trade_return",

#     min_trades=10,
# )

# plt.show()

In [170]:
def plot_mfe_vs_realized_return(
    execution_result: ExecutionResult,
    *,
    figsize: tuple[float, float] = (
        8,
        7,
    ),
) -> tuple[Figure, Axes]:
    trades = execution_result.trades

    if trades.empty:
        raise ValueError(
            "No trades available."
        )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    for side in ["long", "short"]:
        subset = trades.loc[
            trades["side"] == side
        ]

        ax.scatter(
            subset[
                "mfe_return_on_entry"
            ],
            subset[
                "net_return_on_entry"
            ],
            alpha=0.65,
            label=side,
        )

    valid = trades[
        [
            "mfe_return_on_entry",
            "net_return_on_entry",
        ]
    ].dropna()

    if not valid.empty:
        maximum = float(
            max(
                valid.max().max(),
                0.0,
            )
        )

        minimum = float(
            min(
                valid.min().min(),
                0.0,
            )
        )

        ax.plot(
            [minimum, maximum],
            [minimum, maximum],
            linestyle="--",
            linewidth=0.9,
            label="Realized = MFE",
        )

    ax.axhline(
        0,
        linewidth=0.8,
        alpha=0.7,
    )

    ax.axvline(
        0,
        linewidth=0.8,
        alpha=0.7,
    )

    ax.set_xlabel(
        "MFE return on entry"
    )

    ax.set_ylabel(
        "Net realized return"
    )

    ax.set_title(
        "MFE vs Realized Trade Return"
    )

    ax.grid(
        True,
        alpha=0.25,
    )

    ax.legend(loc="best")

    fig.tight_layout()

    return fig, ax

In [171]:
def plot_mae_vs_realized_return(
    execution_result: ExecutionResult,
    *,
    figsize: tuple[float, float] = (
        8,
        7,
    ),
) -> tuple[Figure, Axes]:
    trades = execution_result.trades

    if trades.empty:
        raise ValueError(
            "No trades available."
        )

    fig, ax = plt.subplots(
        figsize=figsize
    )

    for side in ["long", "short"]:
        subset = trades.loc[
            trades["side"] == side
        ]

        ax.scatter(
            subset[
                "mae_return_on_entry"
            ],
            subset[
                "net_return_on_entry"
            ],
            alpha=0.65,
            label=side,
        )

    ax.axhline(
        0,
        linewidth=0.8,
        alpha=0.7,
    )

    ax.axvline(
        0,
        linewidth=0.8,
        alpha=0.7,
    )

    ax.set_xlabel(
        "MAE return on entry"
    )

    ax.set_ylabel(
        "Net realized return"
    )

    ax.set_title(
        "MAE vs Realized Trade Return"
    )

    ax.grid(
        True,
        alpha=0.25,
    )

    ax.legend(loc="best")

    fig.tight_layout()

    return fig, ax

In [172]:
def add_trade_path_efficiency_metrics(
    trades: pd.DataFrame,
) -> pd.DataFrame:
    d = trades.copy()

    realized_return = pd.to_numeric(
        d["net_return_on_entry"],
        errors="coerce",
    )

    mfe_return = pd.to_numeric(
        d["mfe_return_on_entry"],
        errors="coerce",
    )

    mae_return = pd.to_numeric(
        d["mae_return_on_entry"],
        errors="coerce",
    )

    # สัดส่วนกำไรที่ realize ได้
    # เทียบกับกำไรดีที่สุดที่เคยเห็น
    d["mfe_capture_ratio"] = (
        realized_return / mfe_return
    ).where(
        mfe_return > 0
    )

    # กำไรที่คืนกลับไปจาก MFE
    d["mfe_giveback"] = (
        mfe_return - realized_return
    )

    # Outcome เมื่อเทียบกับ range
    # ระหว่าง MAE ถึง MFE
    excursion_range = (
        mfe_return - mae_return
    )

    d["excursion_location"] = (
        realized_return - mae_return
    ) / excursion_range

    # 1.0 = ออกใกล้ MFE
    # 0.0 = ออกใกล้ MAE

    d["winner_after_adverse_excursion"] = (
        (realized_return > 0)
        & (mae_return < 0)
    )

    d["loser_after_positive_mfe"] = (
        (realized_return <= 0)
        & (mfe_return > 0)
    )

    return d

In [173]:
# breakout_4h_execution.trades = (
#     add_trade_path_efficiency_metrics(
#         breakout_4h_execution.trades
#     )
# )

In [174]:
# gave_back_winners = (
#     breakout_4h_execution.trades.loc[
#         breakout_4h_execution.trades[
#             "loser_after_positive_mfe"
#         ]
#     ]
#     .sort_values(
#         "mfe_giveback",
#         ascending=False,
#     )
# )

# display(
#     gave_back_winners[
#         [
#             "trade_id",
#             "side",
#             "net_return_on_entry",
#             "mfe_return_on_entry",
#             "mae_return_on_entry",
#             "mfe_giveback",
#             "holding_bars",
#             "exit_reason",
#         ]
#     ].head(20)
# )

In [175]:
# trade_id = int(
#     gave_back_winners.iloc[0][
#         "trade_id"
#     ]
# )

# fig, axes = plot_trade(
#     breakout_4h_execution,
#     trade_id=trade_id,
#     cfg=trade_cfg,
# )

# plt.show()